In [1]:
# Importar librerías fundamentales para análisis y modelado
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración estética para los gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("¡Entorno de trabajo configurado con éxito!")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

¡Entorno de trabajo configurado con éxito!
Pandas version: 3.0.0
NumPy version: 2.4.2


In [2]:
from sklearn.datasets import make_classification

# Generamos un dataset sintético altamente desbalanceado (simulando transacciones de e-commerce)
# Donde solo el 1% de las transacciones son fraude (anomalías reales)
X, y = make_classification(
    n_samples=10000, 
    n_features=5, 
    n_informative=4, 
    n_redundant=1, 
    weights=[0.99, 0.01], 
    random_state=42
)

# Convertimos los datos a un DataFrame de Pandas con nombres de negocio profesionales
df_transacciones = pd.DataFrame(X, columns=[
    'Monto_Transaccion_Normalizado', 
    'Distancia_Ubicacion_Km', 
    'Frecuencia_Compras_Ultima_Hora', 
    'Edad_Cuenta_Dias', 
    'Dispositivo_Sospechoso_Score'
])

df_transacciones['Es_Fraude'] = y

# Visualizamos las primeras 5 filas
print("Primeras transacciones registradas:")
display(df_transacciones.head())

# Verificamos el desbalance de clases (crucial en detección de anomalías)
conteo_fraudes = df_transacciones['Es_Fraude'].value_counts()
print("\nDistribución de clases (0 = Legítima, 1 = Fraude):")
print(conteo_fraudes)
print(f"\nPorcentaje de fraude en el dataset: {(conteo_fraudes[1] / len(df_transacciones)) * 100:.2f}%")

Primeras transacciones registradas:


,Monto_Transaccion_Normalizado,Distancia_Ubicacion_Km,Frecuencia_Compras_Ultima_Hora,Edad_Cuenta_Dias,Dispositivo_Sospechoso_Score,Es_Fraude
0,-2.299933,3.242833,1.879708,0.066115,1.102833,0
1,-1.707593,0.122327,-0.604671,-0.905535,-0.692637,0
2,-0.916655,-0.569816,-0.719821,-0.782531,-0.772442,0
3,0.805849,-1.691910,-2.878510,-0.472700,-0.355966,0
4,-2.202056,2.670985,-2.196414,0.248894,0.691424,0



Distribución de clases (0 = Legítima, 1 = Fraude):
Es_Fraude
0    9844
1     156
Name: count, dtype: int64

Porcentaje de fraude en el dataset: 1.56%


In [3]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix

# 1. Configuramos el modelo de Detección de Anomalías (Isolation Forest)
# contamination=0.0156 porque sabemos que aproximadamente el 1.56% de nuestros datos son anomalías (fraude)
modelo_anomalias = IsolationForest(
    contamination=0.0156, 
    random_state=42
)

# 2. Entrenamos el modelo con nuestras variables de transacciones (excluyendo la columna de etiqueta real)
X_features = df_transacciones.drop(columns=['Es_Fraude'])
modelo_anomalias.fit(X_features)

# 3. Predicciones del modelo
# Isolation Forest predice 1 para datos normales y -1 para anomalías. 
# Lo transformamos a 0 (legítima) y 1 (fraude) para compararlo con nuestra etiqueta real.
predicciones_raw = modelo_anomalias.predict(X_features)
df_transacciones['Prediccion_Anomalia'] = [1 if p == -1 else 0 for p in predicciones_raw]

# 4. Evaluamos los resultados comparando con el fraude real
print("--- Matriz de Confusión ---")
print(confusion_matrix(df_transacciones['Es_Fraude'], df_transacciones['Prediccion_Anomalia']))

print("\n--- Reporte de Clasificación ---")
print(classification_report(df_transacciones['Es_Fraude'], df_transacciones['Prediccion_Anomalia']))

--- Matriz de Confusión ---
[[9699  145]
 [ 145   11]]

--- Reporte de Clasificación ---
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      9844
           1       0.07      0.07      0.07       156

    accuracy                           0.97     10000
   macro avg       0.53      0.53      0.53     10000
weighted avg       0.97      0.97      0.97     10000



In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# 1. Dividimos nuestros datos en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_features, 
    df_transacciones['Es_Fraude'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df_transacciones['Es_Fraude'] # Mantiene la misma proporción de fraude en ambos grupos
)

# 2. Entrenamos un modelo supervisado (Random Forest)
# Usamos class_weight='balanced' para que el modelo le preste atención a la clase minoritaria (fraude)
modelo_supervisado = RandomForestClassifier(
    n_estimators=100, 
    class_weight='balanced', 
    random_state=42
)

modelo_supervisado.fit(X_train, y_train)

# 3. Evaluamos en el conjunto de prueba que el modelo NUNCA ha visto
y_pred = modelo_supervisado.predict(X_test)

print("--- Matriz de Confusión (Modelo Supervisado) ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- Reporte de Clasificación (Modelo Supervisado) ---")
print(classification_report(y_test, y_pred))

--- Matriz de Confusión (Modelo Supervisado) ---
[[1969    0]
 [  21   10]]

--- Reporte de Clasificación (Modelo Supervisado) ---
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1969
           1       1.00      0.32      0.49        31

    accuracy                           0.99      2000
   macro avg       0.99      0.66      0.74      2000
weighted avg       0.99      0.99      0.99      2000

